# News Data Cleaning

## Import Libraries

In [1]:
# Common Libraries
import numpy as np
import pandas as pd
import os
import sys

# Cleaner output
from tqdm import tqdm
from IPython.display import clear_output

## Project Path
project_path = ".."

## Add the path to text preprocessor
sys.path.append(os.path.abspath(os.path.join(project_path, "lib")))

# Text preprocessing
from preprocessor import clean_text
from scraper import extract_text_from_url

## Import the dataset

In [2]:
news_data = pd.read_csv(os.path.join(project_path, "news_cache/catgorized_data/categorized_news_data2.csv"), sep=",")

## 

In [3]:
news_data.head()

,index,uuid,title,description,keywords,snippet,url,image_url,language,published_at,source,relevance_score,entities,similar,sentiment,text,clean_text,categorical_sentiment_3_class,length
0,0,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,vzphotos istock editorial via getty images sin...,neutral,42
1,1,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,to say that adobe adbe stock has not had a goo...,neutral,259
2,2,9084e5f1-75f5-4f15-aa3d-0676073b4aaf,Global week ahead: The start of a Santa Rally ...,NaN,"STOXX 600, business news",And just like that... December is upon us. It'...,https://www.cnbc.com/2025/11/30/global-week-ah...,https://image.cnbcfm.com/api/v1/image/10823257...,en,2025-11-30T05:10:58.000000Z,cnbc.com,NaN,"[{'symbol': 'M', 'name': ""Macy's, Inc."", 'exch...",[],0.6908,And just like that... December is upon us. It'...,and just like that december is upon us it is b...,positive,493
3,3,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,vzphotos istock editorial via getty images sin...,neutral,42
4,4,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,to say that adobe adbe stock has not had a goo...,neutral,259


## Imputing the `Text`

Since the previous method of fixing the contents of the article iesults in a more worse model, we will refetch the articles

In [4]:
def safe_extract(row):
    url = row["url"]

    if pd.isna(url) or not isinstance(url, str) or url.strip() == "":
        tqdm.write(f"Invalid URL fallback used | URL: {url}")
        tqdm.write(f"{row['title']} {row['description']} {row['snippet']}")
        return f"{row['title']} {row['description']} {row['snippet']}"

    try:
        clear_output(wait=True)
        text = extract_text_from_url(url)

        if text is None or text.strip() == "":
            raise ValueError("Empty scraped text")

        tqdm.write(text)
        return text

    except Exception as e:
        clear_output(wait=True)
        tqdm.write(f"Failed URL fallback used | Err: {e}")
        tqdm.write(f"{row['title']} {row['description']} {row['snippet']}")
        return f"{row['title']} {row['description']} {row['snippet']}"

In [5]:
# tqdm message
tqdm.pandas(desc="Fixing text column")

# apply to the dataset
news_data["text"] = news_data.progress_apply(safe_extract, axis=1)

Fixing text column:  18%|█▊        | 13622/77088 [00:57<04:28, 236.69it/s]


KeyboardInterrupt: 

## Text Cleaning

## Text Cleaning

In [ ]:

categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data_rescraped.csv")
categorized_data_path_folder = os.path.join(project_path,f"news_cache/catgorized_data/")
os.makedirs(categorized_data_path_folder, exist_ok=True)
overwrite_clean_data = True


In [ ]:
# tqdm for cleaner output
tqdm.pandas(desc="Cleaning the Text", unit="news")

# We will cache the data so that it will load faster
if os.path.exists(categorized_data_path) and not overwrite_clean_data:
    print("Loading cached dataset...")
    news_data = pd.read_csv(categorized_data_path)
    print("Cached dataset loaded")

elif os.path.exists(categorized_data_path) and overwrite_clean_data:
    print("Overwriting old data and caching new data...")
    # Clean the data
    news_data["clean_text"] = news_data["text"].progress_apply(
                                                        lambda x: clean_text(
                                                            text = x,
                                                            tokenize=False,
                                                            remove_stop_words= False, # Since we will be using a transformer model, we will not remove stop words since it can be useful for the model to understand the context of the sentence.
                                                            remove_emojis="keep"
                                                            )
                                                        )
    news_data.to_csv(categorized_data_path, index=False)
    print("Done Overwriting old data and caching new data...")

else:
    print("Creating and caching dataset...")
    # Clean the data
    news_data["clean_text"] = news_data["text"].progress_apply(
                                                        lambda x: clean_text(
                                                            text = x,
                                                            tokenize=False,
                                                            remove_stop_words= False,
                                                            remove_emojis="keep"
                                                            )
                                                        )
    news_data.to_csv(categorized_data_path, index=False)
    print("Finished Caching")

Overwriting old data and caching new data...


Cleaning the Text:   0%|          | 0/77088 [00:00<?, ?news/s]

Cleaning the Text: 100%|██████████| 77088/77088 [00:28<00:00, 2737.48news/s]


Done Overwriting old data and caching new data...
